# NordikBank AML — Final Model

**Model:** XGBoost on `features_base.csv` (55 manually engineered AML features)  
**Selection rationale:** XGBoost base won the three-way comparison. tsfresh added complexity without improving either metric.

| Model | Val AUC-ROC | Val PR-AUC | Train-Val Gap |
|---|---|---|---|
| Logistic Regression | 0.8819 | 0.4344 | 0.089 |
| **XGBoost base** | **0.8901** | **0.6173** | 0.110 |
| XGBoost + tsfresh | 0.8790 | 0.4916 | 0.121 |

This notebook: trains the final model · finds the optimal threshold · generates `predictions.csv` (3 columns) · generates all slide assets · saves the model for real-time inference · implements the feedback loop.

## 0. Install & Import

In [ ]:
import subprocess, sys
for pkg in ["xgboost", "shap", "scikit-learn", "pandas", "numpy", "matplotlib"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
print("Ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings, json, pickle
from datetime import datetime
from pathlib import Path
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_recall_curve,
    confusion_matrix, precision_score, recall_score, f1_score,
    ConfusionMatrixDisplay, PrecisionRecallDisplay
)
import xgboost as xgb
import shap

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Notebook: context/nordikbank-modeling/final_model.ipynb
# Data:     data/  (two levels up)
BASE_DIR   = Path("../../")
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TARGET           = "suspicious_activity_confirmed"
NON_FEATURE_COLS = ["customer_id", "split", TARGET]

print(f"Data    : {DATA_DIR.resolve()}")
print(f"Outputs : {OUTPUT_DIR.resolve()}")

## 1. Load & Split

In [ ]:
df_base = pd.read_csv(DATA_DIR / "features_base.csv")
print(f"Shape: {df_base.shape}")
print(df_base["split"].value_counts().to_string())

feature_cols = [c for c in df_base.columns if c not in NON_FEATURE_COLS]

train = df_base[df_base["split"] == "train"]
val   = df_base[df_base["split"] == "val"]
test  = df_base[df_base["split"] == "test"]

X_train  = train[feature_cols];  y_train  = train[TARGET].astype(int)
X_val    = val[feature_cols];    y_val    = val[TARGET].astype(int)
X_test   = test[feature_cols];   test_ids = test["customer_id"]

# scale_pos_weight: matches comparison notebook exactly
n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos
SPW   = n_neg / n_pos

print(f"\nTrain positives : {n_pos}")
print(f"scale_pos_weight: {SPW:.2f}")
print(f"Val positives   : {y_val.sum()}")
print(f"Features        : {len(feature_cols)}")

## 2. Train — Exact Params from Comparison Notebook

In [ ]:
# Params are IDENTICAL to model1 in modeling.ipynb cell 13
model = xgb.XGBClassifier(
    n_estimators          = 300,
    max_depth             = 4,
    learning_rate         = 0.05,
    subsample             = 0.8,
    colsample_bytree      = 0.8,
    scale_pos_weight      = SPW,
    eval_metric           = "auc",
    early_stopping_rounds = 20,   # stops when val AUC stagnates for 20 rounds
    random_state          = RANDOM_STATE,
    use_label_encoder     = False,
    verbosity             = 0,
)

model.fit(
    X_train, y_train,
    eval_set = [(X_val, y_val)],
    verbose  = False,
)

print(f"Done. Best iteration: {model.best_iteration} / 300")

## 3. Evaluate

In [ ]:
val_probs   = model.predict_proba(X_val)[:, 1]
train_probs = model.predict_proba(X_train)[:, 1]
base_rate   = float(y_val.mean())

val_auc    = roc_auc_score(y_val, val_probs)
train_auc  = roc_auc_score(y_train, train_probs)
val_pr_auc = average_precision_score(y_val, val_probs)

print("="*50)
print(f"  Train AUC-ROC : {train_auc:.4f}")
print(f"  Val   AUC-ROC : {val_auc:.4f}   (expected ~0.890)")
print(f"  Val   PR-AUC  : {val_pr_auc:.4f}   (expected ~0.617)")
print(f"  Gap           : {train_auc - val_auc:.4f}")
print(f"  PR lift       : {val_pr_auc/base_rate:.1f}x random ({base_rate:.3f})")

## 4. Threshold Selection

Default 0.5 doesn't work with 4.2% positive rate — almost nothing would be flagged. We use the **F1-optimal threshold on val**: maximises the balance between catching suspects (recall) and not swamping analysts with false alarms (precision).

In [ ]:
precisions, recalls, thresholds = precision_recall_curve(y_val, val_probs)
f1_scores = 2 * precisions[:-1] * recalls[:-1] / (precisions[:-1] + recalls[:-1] + 1e-9)
best_idx  = np.argmax(f1_scores)
THRESHOLD = float(thresholds[best_idx])

val_preds = (val_probs >= THRESHOLD).astype(int)
precision = precision_score(y_val, val_preds, zero_division=0)
recall    = recall_score(y_val, val_preds, zero_division=0)
f1        = f1_score(y_val, val_preds, zero_division=0)

print(f"F1-optimal threshold : {THRESHOLD:.4f}")
print(f"  Precision : {precision:.4f}  — {precision:.0%} of flags are truly suspicious")
print(f"  Recall    : {recall:.4f}  — {recall:.0%} of {y_val.sum()} val suspects caught")
print(f"  F1        : {f1:.4f}")
print(f"  Val flags : {val_preds.sum()} / {len(val_preds)}")

## 5. Slide Assets

In [ ]:
# ── Asset 1: Confusion Matrix + Metrics table ──────────────────────────────────
cm = confusion_matrix(y_val, val_preds)
tn, fp, fn, tp = cm.ravel()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("XGBoost Final Model — Validation Performance", fontsize=14, fontweight="bold", y=1.01)

ConfusionMatrixDisplay(confusion_matrix=cm,
    display_labels=["Clean (0)", "Suspicious (1)"]
).plot(ax=axes[0], colorbar=False, cmap="Purples")
axes[0].set_title(f"Confusion Matrix  (threshold = {THRESHOLD:.3f})", fontsize=11)
for (r,c), lbl in [((0,0),f"TN: {tn}"),((0,1),f"FP: {fp}\n(false alarm)"),
                    ((1,0),f"FN: {fn}\n(missed)"),((1,1),f"TP: {tp}\n(caught)")]:
    axes[0].text(c, r, f"\n{lbl}", ha="center", va="center", fontsize=9, color="gray")

axes[1].axis("off")
rows = [
    ["Val AUC-ROC",   f"{val_auc:.3f}",    "Official panel metric"],
    ["Val PR-AUC",    f"{val_pr_auc:.3f}", f"{val_pr_auc/base_rate:.1f}x random baseline"],
    ["Precision",     f"{precision:.3f}",  f"{precision:.0%} of flags are real suspects"],
    ["Recall",        f"{recall:.3f}",     f"{recall:.0%} of suspects caught"],
    ["F1 Score",      f"{f1:.3f}",         "Balance of precision & recall"],
    ["True Pos (TP)", str(tp),             "Correctly flagged suspicious"],
    ["False Pos (FP)",str(fp),             "Clean customers wrongly flagged"],
    ["False Neg (FN)",str(fn),             "Suspicious customers missed"],
    ["True Neg (TN)", str(tn),             "Correctly cleared clean"],
]
tbl = axes[1].table(cellText=rows, colLabels=["Metric","Value","Meaning"],
                    loc="center", cellLoc="left")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.2, 1.6)
for j in range(3):
    tbl[0,j].set_facecolor("#7B00D4")
    tbl[0,j].set_text_props(color="white", fontweight="bold")
axes[1].set_title("Metrics Summary", fontsize=11)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "confusion_matrix_and_metrics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: confusion_matrix_and_metrics.png")

In [ ]:
# ── Asset 2: Precision-Recall Curve ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
PrecisionRecallDisplay.from_predictions(
    y_val, val_probs,
    name=f"XGBoost base (PR-AUC = {val_pr_auc:.3f})",
    ax=ax, color="#7B00D4"
)
ax.axhline(y=base_rate, color="gray", linestyle="--", linewidth=1,
           label=f"Random baseline ({base_rate:.3f})")
ax.scatter(recalls[best_idx], precisions[best_idx], s=120, zorder=5, color="#FF6B00",
           label=f"Chosen threshold ({THRESHOLD:.3f})  P={precisions[best_idx]:.2f} R={recalls[best_idx]:.2f}")
ax.set_title("Precision-Recall Curve — Validation Set", fontsize=13)
ax.set_xlabel("Recall (fraction of suspicious customers caught)", fontsize=11)
ax.set_ylabel("Precision (fraction of flags that are real)", fontsize=11)
ax.legend(fontsize=9); ax.set_xlim(0,1); ax.set_ylim(0,1.05); ax.grid(alpha=0.3)
ax.text(0.52, 0.10, f"PR-AUC = {val_pr_auc:.3f}\n{val_pr_auc/base_rate:.1f}x random baseline",
        fontsize=10, bbox=dict(boxstyle="round", facecolor="#F3E8FF", alpha=0.8))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "pr_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: pr_curve.png")

In [ ]:
# ── Asset 3: Three-model comparison (numbers from comparison notebook) ─────────
comp = pd.DataFrame({
    "Model"       : ["Logistic\nRegression", "XGBoost\n(base)", "XGBoost\n(+tsfresh)"],
    "Val AUC-ROC" : [0.8819, 0.8901, 0.8790],
    "Val PR-AUC"  : [0.4344, 0.6173, 0.4916],
})
colors = ["#CCCCCC", "#7B00D4", "#CCCCCC"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle("Model Comparison — Validation Set", fontsize=14, fontweight="bold")
for ax, metric in zip(axes, ["Val AUC-ROC", "Val PR-AUC"]):
    bars = ax.bar(comp["Model"], comp[metric], color=colors, edgecolor="white", width=0.5)
    ax.set_title(metric, fontsize=12); ax.set_ylim(0,1)
    ax.grid(axis="y", alpha=0.3); ax.spines[["top","right"]].set_visible(False)
    for bar, v in zip(bars, comp[metric]):
        ax.text(bar.get_x()+bar.get_width()/2, v+0.01, f"{v:.3f}",
                ha="center", va="bottom", fontsize=11, fontweight="bold")
    if metric == "Val PR-AUC":
        ax.axhline(y=base_rate, color="#FF6B00", linestyle="--", linewidth=1.2,
                   label=f"Random ({base_rate:.3f})")
        ax.legend(fontsize=9)
axes[0].annotate("Winner", xy=(1, 0.8901), xytext=(1, 0.96), ha="center",
                 fontsize=10, color="#7B00D4",
                 arrowprops=dict(arrowstyle="->", color="#7B00D4", lw=1.5))
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: model_comparison.png")

In [ ]:
# ── Assets 4-6: SHAP (beeswarm, bar, waterfall) ────────────────────────────────
X_all = pd.concat([X_train, X_val, X_test], ignore_index=True)
all_ids = pd.concat([
    df_base[df_base["split"]=="train"]["customer_id"].reset_index(drop=True),
    df_base[df_base["split"]=="val"]["customer_id"].reset_index(drop=True),
    df_base[df_base["split"]=="test"]["customer_id"].reset_index(drop=True),
], ignore_index=True)

print("Computing SHAP for 1,200 customers...")
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_all)
print("Done.")

# Beeswarm
shap.summary_plot(shap_values, X_all, max_display=20, show=False, plot_size=(10,7))
plt.title("SHAP Feature Importance — All 1,200 Customers", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_summary.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: shap_summary.png")

# Bar chart (cleaner for deck)
shap.summary_plot(shap_values, X_all, plot_type="bar", max_display=15, show=False, plot_size=(9,6))
plt.title("Top 15 Features by Mean |SHAP|", fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_bar.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: shap_bar.png")

# Waterfall for highest-risk val customer
top_idx    = int(np.argmax(val_probs))
top_id     = df_base[df_base["split"]=="val"]["customer_id"].iloc[top_idx]
global_idx = len(X_train) + top_idx
print(f"Top val customer: {top_id}  score={val_probs[top_idx]:.3f}")
shap.waterfall_plot(
    shap.Explanation(
        values=shap_values[global_idx], base_values=explainer.expected_value,
        data=X_all.iloc[global_idx].values, feature_names=X_all.columns.tolist()
    ), max_display=15, show=False
)
plt.title(f"SHAP Waterfall — {top_id}  (score: {val_probs[top_idx]:.3f})", fontsize=11)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shap_waterfall.png", dpi=150, bbox_inches="tight")
plt.show(); print("Saved: shap_waterfall.png")

## 6. Generate predictions.csv

In [ ]:
test_probs = model.predict_proba(X_test)[:, 1]
test_flags = (test_probs >= THRESHOLD).astype(int)

predictions = pd.DataFrame({
    "customer_id"                  : test_ids.values,
    "predicted_probability"        : test_probs,
    "predicted_suspicious_activity": test_flags,
})

assert len(predictions) == 500
assert set(predictions.columns) == {"customer_id","predicted_probability","predicted_suspicious_activity"}
assert predictions["predicted_probability"].between(0,1).all()
assert predictions["predicted_probability"].notna().all()
assert predictions["customer_id"].is_unique
assert set(predictions["customer_id"]) == set(test_ids)
assert predictions["predicted_suspicious_activity"].isin([0,1]).all()

predictions.to_csv(OUTPUT_DIR / "predictions.csv", index=False)
print(f"✓ predictions.csv — {len(predictions)} rows, threshold={THRESHOLD:.4f}")
print(f"  Flagged suspicious: {test_flags.sum()} / 500")
print()
print(predictions.head(10).to_string(index=False))

In [ ]:
# all_predictions.csv (all 1,200 for workbench)
all_probs = model.predict_proba(X_all)[:, 1]
all_flags = (all_probs >= THRESHOLD).astype(int)
pd.DataFrame({
    "customer_id"                  : all_ids,
    "predicted_probability"        : all_probs,
    "predicted_suspicious_activity": all_flags,
}).to_csv(OUTPUT_DIR / "all_predictions.csv", index=False)
print("✓ all_predictions.csv — 1,200 rows")

# prediction_drivers.csv (top-3 SHAP per customer for workbench)
records = []
for i, cid in enumerate(all_ids):
    row_shap = shap_values[i]
    for rank, idx in enumerate(np.argsort(np.abs(row_shap))[::-1][:3], start=1):
        records.append({
            "customer_id"  : cid, "rank": rank,
            "feature_name" : X_all.columns[idx],
            "feature_value": round(float(X_all.iloc[i, idx]), 6),
            "shap_value"   : round(float(row_shap[idx]), 6),
        })
pd.DataFrame(records).to_csv(OUTPUT_DIR / "prediction_drivers.csv", index=False)
print(f"✓ prediction_drivers.csv — {len(records)} rows (1,200 x 3)")

## 7. Save Model for Real-Time Inference

In [ ]:
# JSON (recommended for production)
model.save_model(OUTPUT_DIR / "nordikbank_model.json")

# Pickle (sklearn-compatible)
with open(OUTPUT_DIR / "nordikbank_model.pkl", "wb") as f:
    pickle.dump(model, f)

# Metadata — threshold and feature list MUST travel with the model
metadata = {
    "model_name"      : "XGBoost_base",
    "trained_at"      : datetime.now().isoformat(),
    "val_auc_roc"     : round(val_auc, 4),
    "val_pr_auc"      : round(val_pr_auc, 4),
    "threshold"       : round(THRESHOLD, 6),
    "scale_pos_weight": round(float(SPW), 4),
    "best_iteration"  : int(model.best_iteration),
    "feature_columns" : feature_cols,
    "params": {
        "n_estimators":300, "max_depth":4, "learning_rate":0.05,
        "subsample":0.8, "colsample_bytree":0.8, "random_state":RANDOM_STATE
    }
}
with open(OUTPUT_DIR / "model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

# Verify reload
reloaded = xgb.XGBClassifier()
reloaded.load_model(OUTPUT_DIR / "nordikbank_model.json")
assert np.allclose(reloaded.predict_proba(X_val)[:,1], val_probs, atol=1e-5), "Reload mismatch!"

print("✓ nordikbank_model.json — saved and reload verified")
print("✓ nordikbank_model.pkl")
print("✓ model_metadata.json")
print()
print("Backend usage:")
print("  model = xgb.XGBClassifier()")
print("  model.load_model('nordikbank_model.json')")
print("  meta  = json.load(open('model_metadata.json'))")
print("  prob  = float(model.predict_proba(X)[0, 1])")
print("  flag  = int(prob >= meta['threshold'])")

## 8. Feedback Loop

**This is not reinforcement learning.** True RL requires a reward signal and policy gradient — that's overkill and fragile for a tabular classification problem with rare positives.

What works in production: **periodic supervised retraining on analyst-labelled data.**

```
Model scores customer
  → Analyst reviews in workbench
    → Marks: correct flag / wrong flag / SAR filed / closed
      → Written to feedback_log.csv
        → Weekly: retrain on original + feedback labels (weighted 3x)
          → New model saved as nordikbank_model_v2.json
```

In [ ]:
def retrain_with_feedback(features_path, feedback_path, model_save_path, min_rows=20):
    """
    Retrain incorporating analyst feedback labels.
    Analyst verdicts weighted 3x — they come from actual investigation,
    more reliable than the original compliance audit estimates.

    feedback_log.csv expected columns:
      customer_id, analyst_verdict (0/1), model_score,
      analyst_id, investigation_outcome, timestamp
    """
    features_df = pd.read_csv(features_path)
    feedback_df = pd.read_csv(feedback_path)

    if len(feedback_df) < min_rows:
        print(f"Only {len(feedback_df)} feedback rows — need {min_rows}. Skipping.")
        return None

    feat_cols = [c for c in features_df.columns if c not in NON_FEATURE_COLS]
    train_df  = features_df[features_df["split"] == "train"]
    val_df    = features_df[features_df["split"] == "val"]

    # Original train labels (weight=1)
    X_base = train_df[feat_cols]
    y_base = train_df[TARGET].astype(int)
    w_base = np.ones(len(y_base))

    # Feedback labels (weight=3) — join with features
    fb = feedback_df.merge(features_df[["customer_id"]+feat_cols], on="customer_id", how="inner")
    X_fb = fb[feat_cols]
    y_fb = fb["analyst_verdict"].astype(int)
    w_fb = np.full(len(y_fb), 3.0)

    X_comb = pd.concat([X_base, X_fb], ignore_index=True)
    y_comb = pd.concat([y_base, y_fb], ignore_index=True)
    w_comb = np.concatenate([w_base, w_fb])

    spw_new = (y_comb==0).sum() / (y_comb==1).sum()

    new_model = xgb.XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, scale_pos_weight=spw_new,
        eval_metric="auc", early_stopping_rounds=20,
        random_state=RANDOM_STATE, use_label_encoder=False, verbosity=0,
    )
    new_model.fit(
        X_comb, y_comb, sample_weight=w_comb,
        eval_set=[(val_df[feat_cols], val_df[TARGET].astype(int))],
        verbose=False,
    )

    new_auc = roc_auc_score(val_df[TARGET].astype(int),
                            new_model.predict_proba(val_df[feat_cols])[:,1])
    print(f"Retrained val AUC: {new_auc:.4f}  (baseline: {val_auc:.4f})")
    new_model.save_model(model_save_path)
    print(f"✓ Saved to {model_save_path}")
    return new_model, new_auc


print("retrain_with_feedback() ready.")
print()
print("Key decisions:")
print("  Trigger     : weekly OR >= 20 new feedback rows")
print("  Weighting   : analyst verdicts = 3x original labels")
print("  Threshold   : recompute on val after every retrain")
print("  Versioning  : save as _v2.json, _v3.json — never overwrite")
print("  Risk        : analyst bias — audit feedback patterns before retraining")

## 9. Summary

In [ ]:
print("="*60)
print("FINAL MODEL — COMPLETE")
print("="*60)
print(f"  Val AUC-ROC : {val_auc:.4f}")
print(f"  Val PR-AUC  : {val_pr_auc:.4f}  ({val_pr_auc/base_rate:.1f}x random)")
print(f"  Precision   : {precision:.4f}")
print(f"  Recall      : {recall:.4f}")
print(f"  F1          : {f1:.4f}")
print(f"  Threshold   : {THRESHOLD:.4f}")
print(f"  Test flags  : {test_flags.sum()} / 500")
print()

for fname, desc in [
    ("predictions.csv",                  "Panel submission — 500 rows, 3 columns"),
    ("all_predictions.csv",              "Workbench — all 1,200 customers"),
    ("prediction_drivers.csv",           "Workbench — top-3 SHAP per customer"),
    ("confusion_matrix_and_metrics.png", "Slide asset"),
    ("pr_curve.png",                     "Slide asset"),
    ("model_comparison.png",             "Slide asset"),
    ("shap_summary.png",                 "Slide asset"),
    ("shap_bar.png",                     "Slide asset"),
    ("shap_waterfall.png",               "Slide asset"),
    ("nordikbank_model.json",            "Model for inference"),
    ("nordikbank_model.pkl",             "Model pickle"),
    ("model_metadata.json",             "Threshold + feature list"),
]:
    s = "✓" if (OUTPUT_DIR / fname).exists() else "✗ MISSING"
    print(f"  {s}  {fname:<42} {desc}")